In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,RobustScaler,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings(action='ignore')

In [4]:
plt.style.use('bmh')

In [5]:
import os
print(os.getcwd())

C:\Users\Jay\Desktop\product-return-prediction\notebook


In [6]:
df_dict = pd.read_excel("../data/online_retail_II.xlsx", sheet_name=None)

In [7]:
df_dict.keys()

dict_keys(['Year 2009-2010', 'Year 2010-2011'])

In [8]:
df_dict['Year 2009-2010'].shape, df_dict['Year 2010-2011'].shape

((525461, 8), (541910, 8))

In [9]:
df1 = df_dict['Year 2009-2010']
df2 = df_dict['Year 2010-2011']

In [10]:
df = pd.concat([df1, df2], ignore_index=True)

In [11]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [12]:
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [14]:
df.shape

(1067371, 8)

In [15]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [16]:
df.duplicated().sum()

np.int64(34335)

In [17]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [18]:
df = df.sort_values('InvoiceDate').reset_index(drop=True)

In [19]:
df['is_return'] = ((df['Invoice'].str.startswith('C')) | (df['Quantity'] < 0)).astype(int)

In [20]:
print("Return Distribution:\n", df['is_return'].value_counts(normalize=True))

Return Distribution:
 is_return
0    0.981736
1    0.018264
Name: proportion, dtype: float64


In [21]:
df['Quantity'] = df['Quantity'].abs()
df['Price'] = df['Price'].abs()

In [22]:
df = df[df['Price'] > 0].copy()

In [23]:
df.sample(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_return
111226,500024,84279B,CHERRY BLOSSOM DECORATIVE FLASK,4,2010-03-04 10:49:00,3.75,13879.0,United Kingdom,0
123201,501145,21937,STRAWBERRY PICNIC BAG,3,2010-03-14 12:41:00,2.95,18149.0,United Kingdom,0
682727,550196,22059,CERAMIC STRAWBERRY DESIGN MUG,7,2011-04-15 10:09:00,1.49,14796.0,United Kingdom,0
283849,516960,82551,LAUNDRY 15C METAL SIGN,12,2010-07-26 08:39:00,1.25,12949.0,United Kingdom,0
254442,513948,22383,LUNCH BAG SUKI DESIGN,10,2010-06-29 13:46:00,1.65,14389.0,United Kingdom,0


In [24]:
df['order_value'] = df['Price']*df['Quantity']

In [25]:
df['month'] = df['InvoiceDate'].dt.month

In [26]:
df['hour'] = df['InvoiceDate'].dt.hour

In [27]:
df['day_of_week'] = df['InvoiceDate'].dt.dayofweek

In [28]:
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

In [29]:
df['Customer ID'].isna().sum()

np.int64(236876)

In [30]:
df.dropna(subset=['Customer ID'],inplace=True)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 824293 entries, 0 to 1067370
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      824293 non-null  object        
 1   StockCode    824293 non-null  object        
 2   Description  824293 non-null  object        
 3   Quantity     824293 non-null  int64         
 4   InvoiceDate  824293 non-null  datetime64[ns]
 5   Price        824293 non-null  float64       
 6   Customer ID  824293 non-null  float64       
 7   Country      824293 non-null  object        
 8   is_return    824293 non-null  int64         
 9   order_value  824293 non-null  float64       
 10  month        824293 non-null  int32         
 11  hour         824293 non-null  int32         
 12  day_of_week  824293 non-null  int32         
 13  is_weekend   824293 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int32(3), int64(3), object(4)
memory usage: 84.9+ MB


#### Temporal Split to Prevent Data Leakage

In [32]:
split_point = int(len(df) * 0.8)

In [33]:
train_df = df.iloc[:split_point].copy()
test_df = df.iloc[split_point:].copy()

In [34]:
# 5. Verification
print(f"Total Rows: {len(df)}")
print(f"Training Rows (Past): {len(train_df)} | Dates: {train_df['InvoiceDate'].min()} to {train_df['InvoiceDate'].max()}")
print(f"Testing Rows (Future): {len(test_df)} | Dates: {test_df['InvoiceDate'].min()} to {test_df['InvoiceDate'].max()}")

Total Rows: 824293
Training Rows (Past): 659434 | Dates: 2009-12-01 07:45:00 to 2011-09-09 15:09:00
Testing Rows (Future): 164859 | Dates: 2011-09-09 15:09:00 to 2011-12-09 12:50:00


##### Customer Return Rate

In [35]:
crr = train_df.groupby('Customer ID')['is_return'].mean()

In [36]:
train_df['cust_return_rate'] = train_df['Customer ID'].map(crr)
test_df['cust_return_rate'] = test_df['Customer ID'].map(crr)

In [37]:
print(train_df['cust_return_rate'].mean())
print(test_df['cust_return_rate'].mean())

0.02363238777497066
0.025951914208331883


In [38]:
test_df['cust_return_rate'].isna().sum() ### new customers which has Nan as return rate

np.int64(27851)

In [39]:
### fill that by train mean
train_mean = train_df['is_return'].mean()

In [40]:
test_df['cust_return_rate'] = test_df['cust_return_rate'].fillna(train_mean)

##### Stock Return Rate

In [41]:
stock_rr = train_df.groupby('StockCode')['is_return'].mean()

In [42]:
train_df['stock_return_rate'] = train_df['StockCode'].map(stock_rr)
test_df['stock_return_rate'] = test_df['StockCode'].map(stock_rr)

In [43]:
test_df['stock_return_rate'].fillna(train_mean,inplace=True)

In [44]:
train_df['stock_return_rate'].median()

np.float64(0.012658227848101266)

##### Unique product per Invoice

In [45]:
unique_product = train_df.groupby('Invoice')['StockCode'].nunique().to_dict()

In [46]:
train_df['unique_items'] = train_df['Invoice'].map(unique_product)

In [47]:
test_df['unique_items'] = test_df['Invoice'].map(unique_product)

In [48]:
test_df['unique_items'].isna().sum()  ## new products

np.int64(164810)

In [49]:
test_df['unique_items'].fillna(1,inplace=True)

##### Customer Recency (Days since the customer's last purchase)

In [50]:
curr_date = train_df['InvoiceDate'].max()

In [51]:
last_purchase = train_df.groupby('Customer ID')['InvoiceDate'].max()

In [52]:
cust_recency_map = (curr_date - last_purchase).dt.days.to_dict()

In [53]:
train_df['cust_recency'] = train_df['Customer ID'].map(cust_recency_map).fillna(999)
test_df['cust_recency'] = test_df['Customer ID'].map(cust_recency_map).fillna(999)

##### Top 15 Countries

In [54]:
top_countries = train_df['Country'].value_counts().head(10).index.tolist()

In [55]:
def get_countries(country):
    return country if country in top_countries else "other"

In [56]:
train_df['Country'] = train_df['Country'].apply(get_countries)
test_df['Country'] = test_df['Country'].apply(get_countries)

##### Return Recency (Days since last return)

In [57]:
rr_map = (curr_date - train_df[train_df['is_return']==1].groupby('Customer ID')['InvoiceDate'].max()).dt.days.to_dict()

In [58]:
# Days since last return
train_df['return_recency'] = train_df['Customer ID'].map(rr_map).fillna(365)
test_df['return_recency'] = test_df['Customer ID'].map(rr_map).fillna(365)

##### Order Value Bucket

In [59]:
train_df['order_value_bucket'] = pd.qcut(train_df['order_value'],q=5,labels=False)
test_df['order_value_bucket'] = pd.qcut(test_df['order_value'],q=5,labels=False)

In [60]:
train_df.groupby('order_value_bucket')['is_return'].mean()

order_value_bucket
0    0.031477
1    0.028348
2    0.020795
3    0.013501
4    0.023139
Name: is_return, dtype: float64

##### Quantity Bucket

In [61]:
train_df['quantity_bucket'] = pd.qcut(train_df['Quantity'],q=4,labels=False)
test_df['quantity_bucket'] = pd.qcut(test_df['Quantity'],q=4,labels=False)

In [62]:
train_df.groupby('quantity_bucket')['is_return'].mean()

quantity_bucket
0    0.041450
1    0.020615
2    0.010967
3    0.015381
Name: is_return, dtype: float64

##### Average Unit Price per Customer

In [63]:
avg_price = train_df.groupby('Customer ID')['Price'].mean().to_dict()

In [64]:
train_df['avg_cust_price'] = train_df['Customer ID'].map(avg_price)
test_df['avg_cust_price'] = test_df['Customer ID'].map(avg_price)

In [65]:
price_mean = train_df['Price'].mean()

In [66]:
test_df['avg_cust_price'].fillna(price_mean,inplace=True)

##### Customer Spending Tier

In [67]:
cust_aov = train_df.groupby('Customer ID')['order_value'].mean()

In [68]:
train_df['avg_order_value'] = train_df['Customer ID'].map(cust_aov)
test_df['avg_order_value'] = test_df['Customer ID'].map(cust_aov)

In [69]:
test_df['avg_order_value'].fillna(train_df['order_value'].mean(),inplace=True)

In [70]:
labels = [1,2,3,4,5]

train_df['spending_tier'], bins = pd.qcut(train_df['avg_order_value'],
                                         q=5,
                                         labels=labels,
                                         retbins=True,
                                         duplicates='drop')

test_df['spending_tier'] = pd.cut(test_df['avg_order_value'],
                                 bins=bins,
                                 labels=labels,
                                 include_lowest=True).astype(int)

In [71]:
train_df['spending_tier'] = train_df['spending_tier'].astype(int)

In [72]:
train_df.groupby('spending_tier')['is_return'].mean()

spending_tier
1    0.007695
2    0.012577
3    0.025600
4    0.031286
5    0.041054
Name: is_return, dtype: float64

In [73]:
invoices = train_df.groupby('Customer ID')['Invoice'].nunique()

In [74]:
return_invoices = train_df[train_df['is_return'] == 1].groupby('Customer ID')['Invoice'].nunique()

In [75]:
cust_return_ratio = (return_invoices / invoices).fillna(0).to_dict()

In [76]:
train_df['return_ratio'] = train_df['Customer ID'].map(cust_return_ratio)
test_df['return_ratio'] = test_df['Customer ID'].map(cust_return_ratio)

In [77]:
return_invoices.sum() / invoices.sum()

np.float64(0.1817838552202174)

In [78]:
test_df['return_ratio'].fillna(return_invoices.sum() / invoices.sum(),inplace=True)

##### Items per order ratio

In [79]:
train_df['item_per_order_ratio'] = train_df['unique_items'] / train_df['avg_order_value']
test_df['item_per_order_ratio'] = test_df['unique_items'] / test_df['avg_order_value']

In [80]:
cols_to_drop = ['Invoice','StockCode','Description','InvoiceDate','Customer ID']

In [81]:
train_df.drop(columns=cols_to_drop,inplace=True)
test_df.drop(columns=cols_to_drop,inplace=True)

In [82]:
train_df.shape,test_df.shape

((659434, 21), (164859, 21))

In [83]:
train_df

,Quantity,Price,Country,is_return,order_value,month,hour,day_of_week,is_weekend,cust_return_rate,...,unique_items,cust_recency,return_recency,order_value_bucket,quantity_bucket,avg_cust_price,avg_order_value,spending_tier,return_ratio,item_per_order_ratio
0,12,6.95,United Kingdom,0,83.4,12,7,1,0,0.086957,...,8,66,133.0,4,2,12.413587,37.033696,5,0.200000,0.216019
1,12,6.75,United Kingdom,0,81.0,12,7,1,0,0.086957,...,8,66,133.0,4,2,12.413587,37.033696,5,0.200000,0.216019
2,12,6.75,United Kingdom,0,81.0,12,7,1,0,0.086957,...,8,66,133.0,4,2,12.413587,37.033696,5,0.200000,0.216019
3,48,2.10,United Kingdom,0,100.8,12,7,1,0,0.086957,...,8,66,133.0,4,3,12.413587,37.033696,5,0.200000,0.216019
4,24,1.25,United Kingdom,0,30.0,12,7,1,0,0.086957,...,8,66,133.0,4,3,12.413587,37.033696,5,0.200000,0.216019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859146,8,0.85,United Kingdom,0,6.8,9,15,4,0,0.023256,...,25,0,9.0,1,2,4.789343,29.018974,5,0.236842,0.861505
859147,8,0.85,United Kingdom,0,6.8,9,15,4,0,0.023256,...,25,0,9.0,1,2,4.789343,29.018974,5,0.236842,0.861505
859148,8,0.85,United Kingdom,0,6.8,9,15,4,0,0.023256,...,25,0,9.0,1,2,4.789343,29.018974,5,0.236842,0.861505
859149,4,4.95,United Kingdom,0,19.8,9,15,4,0,0.023256,...,25,0,9.0,3,1,4.789343,29.018974,5,0.236842,0.861505


In [84]:
# train_df['Country'] = le.fit_transform(train_df['Country'])
# test_df['Country'] = le.transform(test_df['Country'])

In [85]:
X_train = train_df.drop(columns=['cust_return_rate', 'return_ratio', 'item_per_order_ratio', 'unique_items','is_return']).fillna(0)
X_test = test_df.drop(columns=['cust_return_rate', 'return_ratio', 'item_per_order_ratio', 'unique_items','is_return']).fillna(0)
y_train = train_df['is_return']
y_test = test_df['is_return']

In [86]:
# cols_to_use = ['unique_items','item_per_order_ratio',
#      'cust_return_rate','stock_return_rate','return_ratio','Quantity','avg_order_value',
#     #  'return_recency','spending_tier','hour','Price','month','avg_cust_price',
#     #  'quantity_bucket','day_of_week','Country'
#     # ]

In [87]:
# test_cols = [
#  'Quantity',
#  'Price',
#  'Country',
#  'month',
#  'hour',
#  'day_of_week',
#  'is_weekend',
#  'order_value',
#  'avg_order_value',
#  'spending_tier'
# ]

In [88]:
# X_train = train_df[test_cols]
# # X_test = test_df[test_cols]
# y_train = train_df['is_return']
# y_test = test_df['is_return']

In [89]:
from imblearn.over_sampling import SMOTE,ADASYN,SMOTENC,RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import make_pipeline,Pipeline
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,OrdinalEncoder

In [90]:
scaler = RobustScaler()
encoder = LabelEncoder()

In [91]:
X_train['Country'] = encoder.fit_transform(X_train['Country'])
X_test['Country'] = encoder.transform(X_test['Country'])

In [92]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [93]:
from imblearn.pipeline import Pipeline

pipeline = Pipeline([
    ('ada', ADASYN(sampling_strategy='minority',random_state=42,n_neighbors=5)),
    ('under', RandomUnderSampler(sampling_strategy='majority'))
])

In [94]:
X_train_resampled,y_train_resampled = pipeline.fit_resample(X_train_scaled,y_train)

In [95]:
X_train_resampled.shape,y_train_resampled.shape

((1287700, 16), (1287700,))

In [96]:
from sklearn.metrics import f1_score,classification_report,confusion_matrix

In [97]:
import optuna

In [98]:
# 1. Split the original, imbalanced scaled data FIRST
# Use the scaled data from cell [95] but BEFORE resampling
from sklearn.model_selection import train_test_split

X_train_honest, X_val_honest, y_train_honest, y_val_honest = train_test_split(
    X_train_scaled, y_train,
    test_size=0.4,
    stratify=y_train,
    random_state=42
)

# 2. Resample ONLY the training portion
# This ensures the validation set (X_val_honest) stays 100% unseen and imbalanced
X_train_res, y_train_res = pipeline.fit_resample(X_train_honest, y_train_honest)

# 3. Re-run your Optuna or XGBoost fit on X_tr_res/y_tr_res
# and evaluate on X_val_honest/y_val_honest

In [99]:
X_train_res.shape,y_train_res.shape,X_val_honest.shape

((771652, 16), (771652,), (263774, 16))

In [100]:
y_train_res.value_counts()

is_return
0    385826
1    385826
Name: count, dtype: int64

### Logistic Regression

In [101]:
from sklearn.linear_model import LogisticRegression

In [102]:
lr_model = LogisticRegression(C=0.01,max_iter=200,random_state=42,n_jobs=-1)

In [103]:
lr_model.fit(X_train_res,y_train_res)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.01
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,200
,multi_class,'deprecated'


In [104]:
lr_y_pred = (lr_model.predict_proba(X_val_honest)[:,1] > 0.7).astype(int)

In [105]:
confusion_matrix(y_val_honest,lr_y_pred)

array([[239324,  18216],
       [  2710,   3524]])

In [106]:
print(classification_report(y_val_honest,lr_y_pred))

              precision    recall  f1-score   support

           0       0.99      0.93      0.96    257540
           1       0.16      0.57      0.25      6234

    accuracy                           0.92    263774
   macro avg       0.58      0.75      0.61    263774
weighted avg       0.97      0.92      0.94    263774



### Random Forest

In [108]:
from sklearn.ensemble import RandomForestClassifier

In [109]:
rf_model = RandomForestClassifier(
    n_estimators=400,bootstrap=True,
    criterion='entropy',max_depth=12,
    n_jobs=-1,random_state=42,
    min_samples_split=3,
)

In [110]:
rf_model.fit(X_train_res,y_train_res)

,n_estimators,400
,criterion,'entropy'
,max_depth,12
,min_samples_split,3
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [111]:
rf_y_pred = (rf_model.predict_proba(X_val_honest)[:,1] > 0.7).astype(int)

In [112]:
confusion_matrix(y_val_honest,rf_y_pred)

array([[252857,   4683],
       [  3138,   3096]])

In [113]:
print(classification_report(y_val_honest,rf_y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98    257540
           1       0.40      0.50      0.44      6234

    accuracy                           0.97    263774
   macro avg       0.69      0.74      0.71    263774
weighted avg       0.97      0.97      0.97    263774



In [125]:
rf_model.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'entropy',
 'max_depth': 12,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 3,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 400,
 'n_jobs': -1,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

### XGBoost

In [115]:
from xgboost import XGBClassifier

In [116]:
xgb_model = XGBClassifier(
    n_estimators= 606, max_depth= 8, 
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    learning_rate= 0.14587621364277914, subsample=0.8162266852324389, 
    colsample_bytree= 0.7057247836490844, gamma= 0.07625629222608954, 
    min_child_weight= 2, reg_alpha= 1.8058595411551832,
    reg_lambda= 9.22761628921285, scale_pos_weight= 5.469033195550583
)

In [117]:
xgb_model.fit(X_train_res,y_train_res)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7057247836490844
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [118]:
y_pred = (xgb_model.predict_proba(X_val_honest)[:,1] > 0.7).astype(int)

In [119]:
print(confusion_matrix(y_val_honest,y_pred))

[[256366   1174]
 [  2421   3813]]


In [120]:
print(classification_report(y_val_honest,y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99    257540
           1       0.76      0.61      0.68      6234

    accuracy                           0.99    263774
   macro avg       0.88      0.80      0.84    263774
weighted avg       0.99      0.99      0.99    263774



In [121]:
features = X_train.columns.to_list()

In [122]:
model.feature_importances_

NameError: name 'model' is not defined

In [ ]:
# 2. Map these names back to the XGBoost booster object
model.get_booster().feature_names = features

# 3. Plot the importance (using 'gain' is usually best for decision-making)
plt.figure(figsize=(10, 8))
plot_importance(model, importance_type='gain', title='Final Model Feature Importance (Gain)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

# Get the probability scores for the 'Return' class (class 1)
y_scores = model.predict_proba(X_val_honest)[:, 1]

# Calculate precision and recall
precision, recall, thresholds = precision_recall_curve(y_val_honest, y_scores)
pr_auc = auc(recall, precision)

# Plotting
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.2f})', color='teal', lw=2)
plt.xlabel('Recall (How many returns did we catch?)')
plt.ylabel('Precision (How accurate were our return predictions?)')
plt.title('Precision-Recall Curve: Product Return Prediction')
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.show()

In [284]:
report_dict = classification_report(y_val_honest,y_pred,output_dict=True)
report_dict

{'0': {'precision': 0.9906996692528824,
  'recall': 0.9955773860371204,
  'f1-score': 0.9931325385205326,
  'support': 257540.0},
 '1': {'precision': 0.7706403544099879,
  'recall': 0.6138915623997433,
  'f1-score': 0.6833928571428571,
  'support': 6234.0},
 'accuracy': 0.9865566735159644,
 'macro avg': {'precision': 0.8806700118314352,
  'recall': 0.8047344742184319,
  'f1-score': 0.8382626978316949,
  'support': 263774.0},
 'weighted avg': {'precision': 0.9854988163684791,
  'recall': 0.9865566735159644,
  'f1-score': 0.9858121916565187,
  'support': 263774.0}}

In [282]:
xgb_params = {
    "n_estimators": 606,
    "max_depth": 8,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "learning_rate": 0.14587621364277914,
    "subsample": 0.8162266852324389,
    "colsample_bytree": 0.7057247836490844,
    "gamma": 0.07625629222608954,
    "min_child_weight": 2,
    "reg_alpha": 1.8058595411551832,
    "reg_lambda": 9.22761628921285,
    "scale_pos_weight": 5.469033195550583
}

In [135]:
import mlflow

In [288]:
# mlflow.set_experiment('Return Prediction')
# mlflow.set_tracking_uri(uri='http://127.0.0.1:5000')

# with mlflow.start_run():
#     mlflow.log_params(xgb_params)
#     mlflow.log_metrics({
#         'precision_class_1': report_dict['1']['precision'],
#         'recall_class_0': report_dict['0']['recall'],
#         'recall_class_1': report_dict['1']['recall'],
#         'f1_score_macro': report_dict['macro avg']['f1-score']
#     })
#     mlflow.xgboost.log_model(model,"XGBoost")

2026/03/21 20:48:32 INFO mlflow.tracking.fluent: Experiment with name 'Return Prediction' does not exist. Creating a new experiment.
2026/03/21 20:48:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run mysterious-quail-916 at: http://127.0.0.1:5000/#/experiments/527746717579408702/runs/f6161373c5994aff864a4299c6e07973
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/527746717579408702


In [126]:
models = [
    (
        'Logistic Regression',
        {'C':0.01,'max_iter':200,'random_state':42,'n_jobs':-1},
        LogisticRegression(),
        (X_train_res,y_train_res),
        (X_val_honest,y_val_honest)
    ),
    (
        'Random Forest',
        {'bootstrap': True,'ccp_alpha': 0.0,'class_weight': None,'criterion': 'entropy','max_depth': 12,
         'max_features': 'sqrt','max_leaf_nodes': None,
         'max_samples': None,'min_impurity_decrease': 0.0,'min_samples_leaf': 1,'min_samples_split': 3,
         'min_weight_fraction_leaf': 0.0,'monotonic_cst': None,'n_estimators': 400,'n_jobs': -1,
         'oob_score': False,'random_state': 42,'verbose': 0,'warm_start': False},
        RandomForestClassifier(),
        (X_train_res,y_train_res),
        (X_val_honest,y_val_honest)
    ),
    (
        'XGBoost',
        {"n_estimators": 606,"max_depth": 8,"objective": "binary:logistic",
        "eval_metric": "logloss","tree_method": "hist","learning_rate": 0.14587621364277914,"subsample": 0.8162266852324389,
        "colsample_bytree": 0.7057247836490844,"gamma": 0.07625629222608954,"min_child_weight": 2,
        "reg_alpha": 1.8058595411551832,"reg_lambda": 9.22761628921285,"scale_pos_weight": 5.469033195550583
        },
        XGBClassifier(),
        (X_train_res,y_train_res),
        (X_val_honest,y_val_honest)
    )
]

In [127]:
reports = []

for model_name,params,model,train_set,val_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = val_set[0]
    y_test = val_set[1]

    model.set_params(**params)
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test,y_pred,output_dict=True)
    reports.append(report)

{'0': {'precision': 0.9927148610614377,
  'recall': 0.9915430612720354,
  'f1-score': 0.9921286151646541,
  'support': 257540.0},
 '1': {'precision': 0.6668706026307739,
  'recall': 0.6993904395251844,
  'f1-score': 0.6827435014093329,
  'support': 6234.0},
 'accuracy': 0.9846383646606565,
 'macro avg': {'precision': 0.8297927318461058,
  'recall': 0.84546675039861,
  'f1-score': 0.8374360582869935,
  'support': 263774.0},
 'weighted avg': {'precision': 0.9850139007429197,
  'recall': 0.9846383646606565,
  'f1-score': 0.9848166481051612,
  'support': 263774.0}}

In [136]:
import mlflow.sklearn
import mlflow.xgboost

In [137]:
mlflow.set_experiment('Product Return Prediction')
mlflow.set_tracking_uri(uri='http://127.0.0.1:5000')

for i,element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({
            'precision_class_1':report['1']['precision'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })

        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/03/22 15:33:50 INFO mlflow.tracking.fluent: Experiment with name 'Product Return Prediction' does not exist. Creating a new experiment.
2026/03/22 15:33:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/22 15:33:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/22 15:34:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/180392047811452637/runs/4a82ca02d2134246ae349d4372c078cc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/180392047811452637


2026/03/22 15:34:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/22 15:34:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/180392047811452637/runs/b9d4e4c52f4d4a04b8b96648a1eba834
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/180392047811452637
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/180392047811452637/runs/155d55447141440f98684990a9d2e567
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/180392047811452637
